# Job-ad preprocessing and exploratory gender-bias detection

This notebook implements a reproducible cleaning workflow and prepares sentence-level material for **human-validated** bias analysis. The workbook's two phrase-label columns are empty, so no existing labels are assumed. Automated dictionary flags and unsupervised clusters below are exploratory aids, not definitive bias judgments or gold-standard annotations.

## Methodological rationale

We adopt the contextual framing of Dikshit, Bouamor, and Habash (2024), who examine job-ad language beyond isolated gender-coded words and organize descriptions using agentic, balanced, and communal groupings. We adapt this idea to sentence-level review and clustering, but do not claim to reproduce their contrastive-learning model. We mask employer names in analysis text only after review, retain identity in separate metadata, and preserve wording and context so that employer-specific templates can be examined rather than silently discarded. This masking is a corpus-specific control, not a demonstrated guarantee against template leakage.

The 2022 studies by Mihaljević et al. and Hu et al. further motivate caution: tool behavior may vary across markets/languages, and bias can occur at text level. Kumar et al. (2025) is related downstream work on ad neutrality in job-ranking systems; this notebook does not implement ranking or mitigation.

**Workflow:** workbook audit → conservative text normalization → employer/contact review and masking → duplicate flags → sentence/bullet units → exploratory lexical flags and human annotation → optional exploratory clustering → descriptive summaries only after annotation.

### References
- Dikshit, M., Bouamor, H., & Habash, N. (2024). [Investigating Gender Bias in STEM Job Advertisements](https://aclanthology.org/2024.gebnlp-1.11/). GeBNLP, 179–189.
- Mihaljević, H., Müller, I., Dill, K., & Yollu-Tok, A. (2022). [Towards gender-inclusive job postings: A data-driven comparison of augmented writing technologies](https://doi.org/10.1371/journal.pone.0274312). *PLOS ONE*, 17(9), e0274312.
- Hu, S. et al. (2022). [Balancing Gender Bias in Job Advertisements With Text-Level Bias Mitigation](https://doi.org/10.3389/fdata.2022.805713). *Frontiers in Big Data*.
- Kumar, D., Masoudian, S., Melchiorre, A. B., & Schedl, M. (2025). [Mitigating Gender Bias in Job Ranking Systems Using Job Advertisement Neutrality](https://aclanthology.org/2025.nlp4pi-1.23/). NLP4PI.

Run cells from top to bottom. Review the employer-candidate file before expecting employer names to be masked. Outputs are written to a separate subfolder; the source workbook is never modified.

In [ ]:
from pathlib import Path
import hashlib
import re
import unicodedata
import warnings

import pandas as pd

# Edit DATA_PATH if the workbook is stored somewhere else.
DATA_PATH = Path.cwd() / "equal-data-set.xlsx"
OUTPUT_DIR = Path.cwd() / "outputs" / "equal_data_preprocessing_bias"
SHEET_NAME = 0
N_CLUSTERS = 6
RUN_EXPLORATORY_CLUSTERING = True
RANDOM_STATE = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Workbook not found at {DATA_PATH}. Set DATA_PATH to the workbook location."
    )

print(f"Input: {DATA_PATH.resolve()}")
print(f"Outputs: {OUTPUT_DIR.resolve()}")

## 1. Load the workbook and audit the supplied fields

The stable ID refers to the source worksheet row so transformations and review annotations can be traced back. Duplicate ads and repeated source links are reported separately; a shared URL is not, by itself, evidence that two ad records should be removed.

In [ ]:
raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME, dtype=object)

def normalize_header(value):
    return re.sub(r"\s+", " ", str(value)).strip()

raw.columns = [normalize_header(column) for column in raw.columns]
required_columns = {
    "Job Advertisement",
    "Source",
    "Gender-Biased Words/Phrase/Statement",
    "Non-Biased Words/Phrase/Statement",
}
missing_columns = required_columns.difference(raw.columns)
if missing_columns:
    raise ValueError(f"Workbook is missing required columns: {sorted(missing_columns)}")

data = raw.copy(deep=True)
data.insert(0, "record_id", [f"row-{row_number:04d}" for row_number in range(2, len(data) + 2)])
data["raw_text"] = data["Job Advertisement"].fillna("").astype(str)
data["source_metadata"] = data["Source"]

text_counts = data["raw_text"].value_counts(dropna=False)
duplicate_text_excess = int(data["raw_text"].duplicated(keep="first").sum())
duplicate_text_groups = int((text_counts > 1).sum())
source_values = data["source_metadata"].fillna("").astype(str).str.strip()
source_counts = source_values[source_values.ne("")].value_counts()
duplicate_source_excess = int(source_values[source_values.ne("")].duplicated().sum())

audit = pd.DataFrame(
    {
        "measure": [
            "records",
            "empty ad texts",
            "missing source values",
            "exact duplicate text rows beyond first occurrence",
            "exact duplicate text groups",
            "duplicate source rows beyond first occurrence",
            "unique non-empty source values",
            "non-empty biased-phrase annotations",
            "non-empty non-biased-phrase annotations",
        ],
        "count": [
            len(data),
            int(data["raw_text"].str.strip().eq("").sum()),
            int(data["source_metadata"].isna().sum()),
            duplicate_text_excess,
            duplicate_text_groups,
            duplicate_source_excess,
            int(source_counts.size),
            int(data["Gender-Biased Words/Phrase/Statement"].notna().sum()),
            int(data["Non-Biased Words/Phrase/Statement"].notna().sum()),
        ],
    }
)
display(audit)
data[["record_id", "raw_text", "source_metadata"]].head(3)

## 2. Conservative normalization and employer review

Unicode is normalized to NFC, line endings and horizontal whitespace are standardized, and bullet markers are made consistent. Case, punctuation, vocabulary, paragraph breaks, and section headings are retained. The first non-empty line is treated only as an **employer candidate** because it may instead be a job title or other heading. Confirm candidates in the review file; unapproved candidates are not masked. Approved canonical employer names and aliases are kept in the metadata output, while their occurrences in analysis text are replaced with `[EMPLOYER]`.

In [ ]:
def normalize_text(value):
    text = unicodedata.normalize("NFC", str(value or ""))
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    normalized_lines = []
    for line in text.split("\n"):
        line = re.sub(r"[\t \u00a0]+", " ", line).strip()
        line = re.sub(r"^\s*[~●▪◦]\s*", "• ", line)
        normalized_lines.append(line)
    # Keep paragraph boundaries but avoid arbitrarily long blank-line runs.
    return re.sub(r"\n{3,}", "\n\n", "\n".join(normalized_lines)).strip()

data["normalized_text"] = data["raw_text"].map(normalize_text)
data["employer_candidate"] = data["normalized_text"].map(
    lambda text: next((line.strip(" •") for line in text.splitlines() if line.strip()), "")
)

review_path = OUTPUT_DIR / "employer_review.csv"
review_columns = ["employer_candidate", "employer_name", "aliases", "approved"]
candidate_counts = (
    data.loc[data["employer_candidate"].ne(""), "employer_candidate"]
    .value_counts()
    .rename_axis("employer_candidate")
    .reset_index(name="record_count")
)

if review_path.exists():
    employer_review = pd.read_csv(review_path, dtype=str, keep_default_na=False)
    missing_review_columns = set(review_columns).difference(employer_review.columns)
    if missing_review_columns:
        raise ValueError(f"Employer review file is missing columns: {sorted(missing_review_columns)}")
    employer_review = employer_review[review_columns]
    new_candidates = candidate_counts[["employer_candidate"]].merge(
        employer_review[["employer_candidate"]], how="left", indicator=True
    )
    new_candidates = new_candidates.loc[new_candidates["_merge"].eq("left_only"), "employer_candidate"]
    if not new_candidates.empty:
        employer_review = pd.concat(
            [
                employer_review,
                pd.DataFrame(
                    {
                        "employer_candidate": new_candidates,
                        "employer_name": "",
                        "aliases": "",
                        "approved": "",
                    }
                ),
            ],
            ignore_index=True,
        )
else:
    employer_review = candidate_counts[["employer_candidate"]].copy()
    employer_review["employer_name"] = ""
    employer_review["aliases"] = ""
    employer_review["approved"] = ""
    employer_review.to_csv(review_path, index=False, encoding="utf-8-sig")

employer_review = employer_review.drop_duplicates("employer_candidate", keep="last")
employer_review.to_csv(review_path, index=False, encoding="utf-8-sig")
display(candidate_counts.head(20))
print(f"Employer candidates: {len(candidate_counts):,}; review file: {review_path}")
print("Review employer_name and aliases, then set approved to True for verified employer rows.")

### Employer review checkpoint

Open `employer_review.csv` from the output folder. For each verified employer, enter its canonical name, any alternate spellings in `aliases` separated by `|`, and `True` in `approved`. Leave uncertain rows unapproved. Then rerun this and subsequent cells. Employer names are not inferred as verified solely from their position in the ad.

In [ ]:
def is_approved(value):
    return str(value).strip().casefold() in {"true", "1", "yes", "y"}

approved_review = employer_review.loc[
    employer_review["approved"].map(is_approved)
    & employer_review["employer_name"].astype(str).str.strip().ne("")
].copy()

alias_to_employer = {}
for _, employer_row in approved_review.iterrows():
    canonical = employer_row["employer_name"].strip()
    aliases = [canonical, employer_row["employer_candidate"].strip()]
    aliases.extend(
        alias.strip()
        for alias in re.split(r"[|;]", str(employer_row["aliases"]))
        if alias.strip()
    )
    for alias in aliases:
        alias_to_employer[alias.casefold()] = canonical

def stable_employer_key(employer_name):
    digest = hashlib.sha256(employer_name.casefold().encode("utf-8")).hexdigest()[:12]
    return f"employer-{digest}"

def mask_verified_employers(text):
    for alias in sorted(alias_to_employer, key=len, reverse=True):
        pattern = re.compile(r"(?<!\w)" + re.escape(alias) + r"(?!\w)", flags=re.IGNORECASE)
        text = pattern.sub("[EMPLOYER]", text)
    return text

data["employer_name_metadata"] = data["employer_candidate"].map(
    lambda candidate: alias_to_employer.get(candidate.casefold(), "")
)
data["employer_key"] = data["employer_name_metadata"].map(
    lambda name: stable_employer_key(name) if name else ""
)
data["text_after_employer_mask"] = data["normalized_text"].map(mask_verified_employers)

print(f"Approved employers: {len(approved_review):,}")
print(f"Verified aliases available for masking: {len(alias_to_employer):,}")
if not alias_to_employer:
    print("No approved employer mapping was found; employer names remain unmasked pending review.")

## 3. Contact-detail masking and duplicate flags

Email addresses and URLs are replaced with typed placeholders; phone numbers are masked when explicitly labelled or presented as a phone-like standalone line. Clearly labelled address/contact lines are replaced, while ambiguous geographic or street-like lines are flagged for manual inspection rather than deleted on a broad keyword guess. Original `Source` values stay separate as provenance metadata. Exact duplicate text is grouped and retained for review.

In [ ]:
EMAIL_RE = re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.IGNORECASE)
URL_RE = re.compile(r"\b(?:https?://|www\.)\S+", re.IGNORECASE)
PHONE_LABEL_RE = re.compile(
    r"(?i)(\b(?:tel(?:ephone)?|mobile|phone|contact)\s*(?:no\.?|number)?\s*[:\-]?\s*)"
    r"(\+?\d[\d\s().\-]{6,}\d)"
)
PHONE_LINE_RE = re.compile(r"^\s*(?:[•~-]\s*)?\+?\d[\d\s().\-]{7,}\d\s*$")
ADDRESS_LABEL_RE = re.compile(
    r"(?i)^\s*(?:address|office address|head office|visit us at)\s*:\s*.*$"
)
ADDRESS_CUE_RE = re.compile(
    r"(?i)\b(?:street|st\.|road|rd\.|avenue|ave\.|block|blk\.?|lot|phase|barangay|brgy\.)\b"
)

def mask_contact_details(text):
    text = EMAIL_RE.sub("[EMAIL]", text)
    text = URL_RE.sub("[URL]", text)
    text = PHONE_LABEL_RE.sub(lambda match: match.group(1) + "[PHONE]", text)
    cleaned_lines = []
    for line in text.splitlines():
        if ADDRESS_LABEL_RE.match(line):
            cleaned_lines.append("[CONTACT_ADDRESS]")
        elif PHONE_LINE_RE.match(line):
            cleaned_lines.append("[PHONE]")
        else:
            cleaned_lines.append(line)
    return "\n".join(cleaned_lines)

data["analysis_text"] = data["text_after_employer_mask"].map(mask_contact_details)
data["exact_duplicate_group"] = data["normalized_text"].map(
    lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest() if text else ""
)
data["is_exact_duplicate"] = data["normalized_text"].duplicated(keep=False) & data["normalized_text"].ne("")
data["has_address_review_cue"] = data["analysis_text"].map(
    lambda text: bool(ADDRESS_CUE_RE.search(text))
)

print(f"Records in exact-duplicate groups: {int(data['is_exact_duplicate'].sum()):,}")
print(f"Records with possible address/location lines to review: {int(data['has_address_review_cue'].sum()):,}")
display(
    data.loc[data["is_exact_duplicate"], ["record_id", "exact_duplicate_group", "employer_candidate"]]
    .sort_values("exact_duplicate_group")
    .head(20)
)

### Sample of unmasked and masked text

A reproducible sample (seed 42) of advertisements in which masking changed the text. *Unmasked* is the standardized text before masking; *masked* is `analysis_text`. Employer names are replaced only for rows approved in `employer_review.csv`; until then only contact details are masked. The unmasked column contains real contact details, so treat saved outputs of this cell as restricted.

In [ ]:
changed = data[data["normalized_text"] != data["analysis_text"]]
print(f"Ads changed by masking: {len(changed):,} of {len(data):,}")
for tag in ["[EMPLOYER]", "[EMAIL]", "[URL]", "[PHONE]", "[CONTACT_ADDRESS]"]:
    print(f"  {tag}: {int(data['analysis_text'].str.contains(tag, regex=False).sum()):,} ads")

sample = changed.sample(n=min(3, len(changed)), random_state=RANDOM_STATE)
for _, row in sample.iterrows():
    print("\n" + "=" * 90)
    print(f"{row['record_id']} | employer candidate (metadata): {row['employer_candidate']}")
    print("-" * 40 + " UNMASKED " + "-" * 40)
    print(row["normalized_text"][:1200])
    print("-" * 41 + " MASKED " + "-" * 41)
    print(row["analysis_text"][:1200])

## 4. Sentence/bullet segmentation and exploratory lexical flags

Each line is divided at sentence-ending punctuation where possible; the original order and record relationship are retained. The dictionaries are editable starter signals, not validated Philippine-English/Filipino gender lexicons. Words such as “assertive” or “supportive” are not inherently biased; a hit is only a prompt for contextual human review. Sentence annotations should capture the evidence span and assign one of `agentic`, `communal`, `balanced`, `other`, or `uncertain`. Use `balanced` only when the sentence genuinely combines relevant cues; do not use it as a default neutral class.

In [ ]:
# Illustrative, editable review cues; do not interpret as a validated lexicon.
AGENTIC_CUES = [
    "ambitious", "assertive", "competitive", "decisive", "independent",
    "leadership", "leader", "dominant", "confident", "determined",
]
COMMUNAL_CUES = [
    "collaborative", "supportive", "empathetic", "nurturing", "caring",
    "team player", "interpersonal", "cooperative", "compassionate",
]
EXPLICIT_GENDER_CUES = ["male", "female", "men", "women", "man", "woman", "boys", "girls"]

def compile_phrase_pattern(phrases):
    alternatives = "|".join(re.escape(phrase) for phrase in sorted(phrases, key=len, reverse=True))
    return re.compile(r"(?<!\w)(?:" + alternatives + r")(?!\w)", flags=re.IGNORECASE)

AGENTIC_PATTERN = compile_phrase_pattern(AGENTIC_CUES)
COMMUNAL_PATTERN = compile_phrase_pattern(COMMUNAL_CUES)
GENDER_PATTERN = compile_phrase_pattern(EXPLICIT_GENDER_CUES)

def split_units(text):
    units = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        units.extend(part.strip() for part in re.split(r"(?<=[.!?])\s+", line) if part.strip())
    return units

sentence_rows = []
for row in data[["record_id", "analysis_text", "employer_name_metadata", "employer_key"]].itertuples(index=False):
    units = split_units(row.analysis_text)
    for unit_number, unit_text in enumerate(units, start=1):
        sentence_rows.append(
            {
                "record_id": row.record_id,
                "sentence_id": f"{row.record_id}-u{unit_number:04d}",
                "unit_index": unit_number,
                "sentence_text": unit_text,
                "agentic_cue_hits": " | ".join(match.group(0) for match in AGENTIC_PATTERN.finditer(unit_text)),
                "communal_cue_hits": " | ".join(match.group(0) for match in COMMUNAL_PATTERN.finditer(unit_text)),
                "explicit_gender_hits": " | ".join(match.group(0) for match in GENDER_PATTERN.finditer(unit_text)),
                "human_label": "",
                "evidence_span": "",
                "reviewer_notes": "",
            }
        )

sentences = pd.DataFrame(sentence_rows)
if sentences.empty:
    raise ValueError("No sentence/bullet units were produced; inspect the source workbook and cleaning steps.")

annotation_path = OUTPUT_DIR / "sentence_annotation_review.csv"
annotation_fields = ["human_label", "evidence_span", "reviewer_notes"]
if annotation_path.exists():
    prior_annotations = pd.read_csv(annotation_path, dtype=str, keep_default_na=False)
    required_annotation_columns = {"sentence_id", "sentence_text", *annotation_fields}
    missing_annotation_columns = required_annotation_columns.difference(prior_annotations.columns)
    if missing_annotation_columns:
        raise ValueError(f"Annotation file is missing columns: {sorted(missing_annotation_columns)}")
    stale = prior_annotations.merge(
        sentences[["sentence_id", "sentence_text"]], on="sentence_id", how="left", suffixes=("_old", "_current")
    )
    changed = stale["sentence_text_current"].notna() & stale["sentence_text_old"].ne(stale["sentence_text_current"])
    if changed.any():
        warnings.warn(f"{int(changed.sum())} reviewed sentence IDs now have changed text; check annotations before analysis.")
    annotations = prior_annotations[["sentence_id", *annotation_fields]].drop_duplicates("sentence_id", keep="last")
    sentences = sentences.drop(columns=annotation_fields).merge(annotations, on="sentence_id", how="left")
    sentences[annotation_fields] = sentences[annotation_fields].fillna("")

sentences.to_csv(annotation_path, index=False, encoding="utf-8-sig")
print(f"Sentence/bullet units: {len(sentences):,}; review file: {annotation_path}")
print("Fill human_label and evidence_span after review; automated cue columns are not labels.")
display(sentences[["record_id", "sentence_text", "agentic_cue_hits", "communal_cue_hits", "explicit_gender_hits"]].head(12))

## 5. Optional unsupervised sentence clustering

TF-IDF plus K-means provides a reproducible baseline for grouping similar sentences. It does not use bias labels, and cluster IDs have no inherent agentic/communal meaning. Inspect representative examples and terms; cluster labels should only be assigned by a documented human review. This baseline differs from the contrastive-learning approach in Dikshit et al. (2024).

In [ ]:
if RUN_EXPLORATORY_CLUSTERING:
    from sklearn.cluster import KMeans
    from sklearn.feature_extraction.text import TfidfVectorizer

    if N_CLUSTERS < 2 or N_CLUSTERS >= len(sentences):
        raise ValueError("N_CLUSTERS must be at least 2 and smaller than the number of sentence units.")

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)
    sentence_matrix = vectorizer.fit_transform(sentences["sentence_text"])
    if sentence_matrix.shape[1] == 0:
        raise ValueError("TF-IDF produced no usable features; inspect language/content or adjust vectorizer settings.")

    cluster_model = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
    sentences["exploratory_cluster"] = cluster_model.fit_predict(sentence_matrix)
    terms = vectorizer.get_feature_names_out()
    top_terms = {}
    for cluster_number, weights in enumerate(cluster_model.cluster_centers_):
        top_indices = weights.argsort()[::-1][:10]
        top_terms[cluster_number] = ", ".join(terms[index] for index in top_indices)

    cluster_summary = (
        sentences.groupby("exploratory_cluster")
        .agg(sentence_count=("sentence_id", "size"), representative_text=("sentence_text", "first"))
        .reset_index()
    )
    cluster_summary["top_tfidf_terms"] = cluster_summary["exploratory_cluster"].map(top_terms)
    display(cluster_summary)
    cluster_summary.to_csv(OUTPUT_DIR / "exploratory_cluster_summary.csv", index=False, encoding="utf-8-sig")
else:
    sentences["exploratory_cluster"] = pd.NA
    print("Clustering skipped by configuration.")

## 6. Descriptive summaries and exports

This section reports human annotation counts only when labels have been entered. It does not infer ad-level bias from dictionary hits or cluster membership. No causal claim about applicant behavior or labor-market outcomes can be made from this text-only workflow.

In [ ]:
valid_labels = {"agentic", "communal", "balanced", "other", "uncertain"}
submitted_labels = sentences["human_label"].astype(str).str.strip().str.casefold()
invalid_labels = sorted(set(submitted_labels) - {""} - valid_labels)
if invalid_labels:
    raise ValueError(f"Unexpected human_label values: {invalid_labels}. Allowed values: {sorted(valid_labels)}")

annotated = sentences.loc[submitted_labels.ne("")].copy()
if annotated.empty:
    print("No human labels entered yet; ad-level bias summaries are intentionally not produced.")
else:
    print(f"Human-labelled units: {len(annotated):,} of {len(sentences):,}")
    display(annotated["human_label"].value_counts().rename_axis("human_label").to_frame("sentence_count"))
    ad_label_summary = (
        annotated.groupby(["record_id", "human_label"]).size().unstack(fill_value=0)
    )
    ad_label_summary["annotated_sentence_count"] = ad_label_summary.sum(axis=1)
    ad_label_summary = ad_label_summary.reset_index()
    ad_label_summary.to_csv(OUTPUT_DIR / "human_annotation_ad_summary.csv", index=False, encoding="utf-8-sig")
    display(ad_label_summary.head())

processed_records = data[
    [
        "record_id",
        "raw_text",
        "normalized_text",
        "analysis_text",
        "source_metadata",
        "employer_candidate",
        "employer_name_metadata",
        "employer_key",
        "exact_duplicate_group",
        "is_exact_duplicate",
        "has_address_review_cue",
    ]
].copy()
processed_records.to_csv(OUTPUT_DIR / "processed_job_ads.csv", index=False, encoding="utf-8-sig")
sentences.to_csv(annotation_path, index=False, encoding="utf-8-sig")

print(f"Saved processed records: {OUTPUT_DIR / 'processed_job_ads.csv'}")
print(f"Saved sentence review/annotation file: {annotation_path}")
print("The source workbook remains unchanged. Treat exported raw/source/employer metadata as restricted research data.")

## Limitations and researcher approval points

- Employer candidates come from the first non-empty line and require manual verification; approved aliases only are masked. Review files before interpreting results.
- Contact masking is rule-based. Review `has_address_review_cue` rows and sample transformed text for missed or over-masked details.
- Sentence splitting may split abbreviations or leave headings/fragments. Inspect segmentation quality before annotation.
- Starter cue lists are not validated for the dataset's language varieties or local context. Expand and document them only after a literature review and annotator agreement process.
- Clusters are sensitive to vectorizer settings and cluster count; cluster IDs are not semantic categories. Compare examples and stability before using any grouping in substantive analysis.
- Develop an annotation guide, double-annotate a sample, adjudicate disagreements, and report agreement before treating human labels as research outcomes.
- This notebook supports preprocessing and exploratory detection only. It does not establish causality, estimate hiring effects, or reproduce the full 2024 contrastive-learning model.

**Before final analysis:** approve the employer review, inspect privacy-cleaning samples, finalize the annotation scheme and lexicon, and decide whether clustering is appropriate for the research question.